# Module 4.1: Crude Oil Fundamentals Modeling

## Learning Objectives
By completing this notebook, you will:
1. Extract crude oil supply/demand data from EIA reports using LLMs
2. Build a structured fundamental balance model
3. Analyze the relationship between fundamentals and prices
4. Generate forward-looking fundamental assessments

## Prerequisites
- Understanding of commodity fundamentals (supply/demand balances)
- Completion of Module 1 (Report Processing)
- Basic knowledge of oil markets

## Estimated Time: 90 minutes

---

## Setup & Imports

We'll use Anthropic's Claude for extraction and LangChain for orchestration.

In [ ]:
# Purpose: Import required libraries and configure environment
# Key Concept: Setting up LLM access and data processing tools

import os
import json
import requests
from datetime import datetime, timedelta
from typing import Dict, List, Optional
from dataclasses import dataclass, asdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from langchain_anthropic import ChatAnthropic
from langchain.prompts import ChatPromptTemplate
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
sns.set_style('whitegrid')

# Random seed for reproducibility
np.random.seed(42)

print("✓ Imports complete")

## 1. Understanding Crude Oil Fundamentals

### Why Fundamentals Matter

Crude oil prices are ultimately driven by **supply and demand dynamics**:

**Supply Components:**
- Production (OPEC, US shale, other)
- Imports
- Stock draws (inventory releases)

**Demand Components:**
- Refinery runs (crude inputs)
- Exports
- Stock builds (inventory additions)

**Balance:**
```
Surplus = Supply - Demand
```

A surplus indicates oversupply (bearish), while a deficit indicates tight markets (bullish).

### LLM Value Proposition

Traditional fundamental analysis relies on structured data tables. LLMs enable:
1. Extracting data from prose (EIA commentary)
2. Parsing qualitative factors (geopolitics, weather)
3. Integrating multiple report types
4. Generating forward assessments

## 2. Defining the Fundamental Data Model

We'll create structured models for crude oil fundamentals.

In [ ]:
# Purpose: Define Pydantic models for structured fundamental data
# Key Concept: Type-safe data structures for LLM extraction

class ProductionData(BaseModel):
    """Crude oil production data."""
    
    region: str = Field(description="Geographic region (e.g., 'US', 'OPEC', 'Russia')")
    value: float = Field(description="Production volume")
    unit: str = Field(description="Unit of measurement (e.g., 'million_bpd')")
    change_wow: Optional[float] = Field(default=None, description="Week-over-week change")
    change_yoy: Optional[float] = Field(default=None, description="Year-over-year change")
    notes: Optional[str] = Field(default=None, description="Qualitative context")


class InventoryData(BaseModel):
    """Crude oil inventory/storage data."""
    
    location: str = Field(description="Storage location (e.g., 'Cushing', 'SPR', 'Total US')")
    level: float = Field(description="Current inventory level")
    unit: str = Field(description="Unit (e.g., 'million_barrels')")
    change_wow: float = Field(description="Weekly change")
    vs_5yr_avg: Optional[float] = Field(default=None, description="Difference from 5-year average")
    vs_5yr_pct: Optional[float] = Field(default=None, description="Percent above/below 5-year average")


class DemandData(BaseModel):
    """Crude oil demand/consumption data."""
    
    category: str = Field(description="Demand category (e.g., 'refinery_inputs', 'exports')")
    value: float = Field(description="Demand volume")
    unit: str = Field(description="Unit of measurement")
    change_wow: Optional[float] = Field(default=None, description="Week-over-week change")
    utilization_rate: Optional[float] = Field(default=None, description="Refinery utilization %")


class CrudeFundamentals(BaseModel):
    """Complete crude oil fundamental data structure."""
    
    report_date: str = Field(description="Report publication date (YYYY-MM-DD)")
    data_period: str = Field(description="Period covered by data")
    production: List[ProductionData] = Field(description="Production data points")
    inventories: List[InventoryData] = Field(description="Inventory data points")
    demand: List[DemandData] = Field(description="Demand data points")
    qualitative_factors: List[str] = Field(description="Key qualitative factors mentioned")
    outlook: str = Field(description="Forward-looking assessment from report")


print("✓ Fundamental data models defined")

## 3. Extracting Fundamentals from EIA Reports

The EIA Weekly Petroleum Status Report (WPSR) is the gold standard for US oil fundamentals.

In [ ]:
# Purpose: Create LLM-based fundamental data extractor
# Key Concept: Converting unstructured reports to structured data

class FundamentalExtractor:
    """Extract fundamental data from EIA reports using LLMs."""
    
    def __init__(self, api_key: Optional[str] = None):
        self.llm = ChatAnthropic(
            model="claude-3-5-sonnet-20241022",
            api_key=api_key or os.environ.get("ANTHROPIC_API_KEY"),
            temperature=0
        )
        self.parser = PydanticOutputParser(pydantic_object=CrudeFundamentals)
        
    def extract(self, report_text: str) -> CrudeFundamentals:
        """
        Extract structured fundamental data from report text.
        
        Parameters
        ----------
        report_text : str
            Raw text from EIA WPSR or similar report
            
        Returns
        -------
        CrudeFundamentals
            Structured fundamental data
        """
        # Step 1: Build extraction prompt
        prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert commodity analyst extracting fundamental data 
from oil market reports. Extract all quantitative data points with precise values and units.

Focus on:
- Production levels and changes
- Inventory levels and weekly changes
- Demand indicators (refinery runs, utilization)
- Comparisons to historical averages
- Qualitative factors (geopolitics, weather, maintenance)

{format_instructions}"""),
            ("user", "Extract fundamental data from this EIA report:\n\n{report_text}")
        ])
        
        # Step 2: Execute extraction
        chain = prompt | self.llm
        response = chain.invoke({
            "report_text": report_text,
            "format_instructions": self.parser.get_format_instructions()
        })
        
        # Step 3: Parse structured output
        fundamentals = self.parser.parse(response.content)
        
        return fundamentals


# Initialize extractor
extractor = FundamentalExtractor()

print("✓ Fundamental extractor initialized")

Let's test the extractor with a sample EIA report excerpt.

In [ ]:
# Purpose: Test fundamental extraction with sample report
# Key Concept: Real-world extraction from unstructured text

# Sample EIA Weekly Petroleum Status Report excerpt
sample_report = """
EIA Weekly Petroleum Status Report - December 27, 2023
Data for Week Ending December 22, 2023

U.S. crude oil production averaged 13.3 million barrels per day (b/d) during the week 
ending December 22, 2023, an increase of 100,000 b/d from the previous week. Production 
is up 1.2 million b/d compared to the same period last year.

U.S. commercial crude oil inventories (excluding Strategic Petroleum Reserve) increased 
by 8.9 million barrels from the previous week. At 439.7 million barrels, inventories are 
about 4% below the five-year average for this time of year.

Crude oil stocks at Cushing, Oklahoma increased by 1.8 million barrels to 24.5 million 
barrels, remaining below the 5-year average of 32 million barrels for this period.

Crude oil inputs to refineries averaged 16.2 million b/d during the week, down 112,000 b/d 
from the previous week. Refinery utilization stood at 90.8%, down 0.6 percentage points 
from the prior week.

U.S. crude oil exports averaged 4.8 million b/d, up from 4.2 million b/d the previous week.

Key factors: Mild winter weather has reduced heating oil demand. Several Gulf Coast refineries 
are undergoing planned maintenance. OPEC+ production cuts remain in effect through Q1 2024.

Outlook: Near-term fundamentals appear balanced, with seasonal demand softness offset by 
production discipline. Geopolitical risks remain elevated given Middle East tensions.
"""

# Extract fundamentals
fundamentals = extractor.extract(sample_report)

# Display structured data
print("=== PRODUCTION DATA ===")
for prod in fundamentals.production:
    print(f"  {prod.region}: {prod.value} {prod.unit}")
    if prod.change_yoy:
        print(f"    YoY Change: +{prod.change_yoy} {prod.unit}")

print("\n=== INVENTORY DATA ===")
for inv in fundamentals.inventories:
    print(f"  {inv.location}: {inv.level} {inv.unit}")
    print(f"    Weekly Change: {inv.change_wow:+.1f} {inv.unit}")
    if inv.vs_5yr_pct:
        print(f"    vs 5-yr avg: {inv.vs_5yr_pct:+.1f}%")

print("\n=== DEMAND DATA ===")
for dem in fundamentals.demand:
    print(f"  {dem.category}: {dem.value} {dem.unit}")
    if dem.utilization_rate:
        print(f"    Utilization: {dem.utilization_rate:.1f}%")

print("\n=== QUALITATIVE FACTORS ===")
for factor in fundamentals.qualitative_factors:
    print(f"  - {factor}")

print(f"\n=== OUTLOOK ===")
print(f"  {fundamentals.outlook}")

## 4. Building the Supply/Demand Balance

Now we'll calculate the fundamental balance and visualize the components.

In [ ]:
# Purpose: Calculate supply/demand balance from fundamental data
# Key Concept: Quantifying market tightness/oversupply

@dataclass
class SupplyDemandBalance:
    """Supply/demand balance calculation."""
    
    date: str
    total_supply: float
    total_demand: float
    balance: float  # Supply - Demand
    inventory_change: float
    days_of_supply: float
    interpretation: str


def calculate_balance(fundamentals: CrudeFundamentals, 
                     daily_demand_estimate: float = 20.0) -> SupplyDemandBalance:
    """
    Calculate supply/demand balance from fundamental data.
    
    Parameters
    ----------
    fundamentals : CrudeFundamentals
        Extracted fundamental data
    daily_demand_estimate : float
        Estimated daily demand (million b/d)
        
    Returns
    -------
    SupplyDemandBalance
        Calculated balance metrics
    """
    # Step 1: Sum supply components
    total_production = sum(p.value for p in fundamentals.production)
    
    # Step 2: Sum demand components (refinery inputs + exports)
    total_demand = sum(d.value for d in fundamentals.demand)
    
    # Step 3: Calculate balance
    balance = total_production - total_demand
    
    # Step 4: Get inventory change
    inventory_change = sum(i.change_wow for i in fundamentals.inventories)
    
    # Step 5: Calculate days of supply (current inventory / daily demand)
    total_inventory = sum(i.level for i in fundamentals.inventories if "Total" in i.location)
    days_of_supply = total_inventory / daily_demand_estimate if total_inventory > 0 else 0
    
    # Step 6: Interpret balance
    if balance > 0.5:
        interpretation = "OVERSUPPLY: Production exceeds demand significantly (bearish)"
    elif balance < -0.5:
        interpretation = "DEFICIT: Demand exceeds production (bullish)"
    else:
        interpretation = "BALANCED: Supply and demand roughly in equilibrium"
    
    return SupplyDemandBalance(
        date=fundamentals.report_date,
        total_supply=total_production,
        total_demand=total_demand,
        balance=balance,
        inventory_change=inventory_change,
        days_of_supply=days_of_supply,
        interpretation=interpretation
    )


# Calculate balance for our sample
balance = calculate_balance(fundamentals)

print("=== SUPPLY/DEMAND BALANCE ===")
print(f"Date: {balance.date}")
print(f"Total Supply: {balance.total_supply:.2f} million b/d")
print(f"Total Demand: {balance.total_demand:.2f} million b/d")
print(f"Balance: {balance.balance:+.2f} million b/d")
print(f"Inventory Change: {balance.inventory_change:+.2f} million barrels")
print(f"Days of Supply: {balance.days_of_supply:.1f} days")
print(f"\nInterpretation: {balance.interpretation}")

## 5. Visualizing the Fundamental Balance

In [ ]:
# Purpose: Visualize supply/demand components and balance
# Key Concept: Making fundamentals interpretable through visualization

def visualize_fundamentals(fundamentals: CrudeFundamentals, balance: SupplyDemandBalance):
    """
    Create comprehensive fundamental visualization.
    
    Parameters
    ----------
    fundamentals : CrudeFundamentals
        Fundamental data
    balance : SupplyDemandBalance
        Balance calculation
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Supply/Demand Waterfall
    ax1 = axes[0, 0]
    categories = ['Production', 'Refinery\nInputs', 'Exports', 'Balance']
    values = [
        balance.total_supply,
        -sum(d.value for d in fundamentals.demand if 'refinery' in d.category.lower()),
        -sum(d.value for d in fundamentals.demand if 'export' in d.category.lower()),
        balance.balance
    ]
    colors = ['green', 'red', 'red', 'blue' if balance.balance < 0 else 'orange']
    ax1.bar(categories, values, color=colors, alpha=0.7)
    ax1.axhline(y=0, color='black', linestyle='--', linewidth=1)
    ax1.set_ylabel('Million b/d', fontsize=12)
    ax1.set_title('Supply/Demand Waterfall', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Inventory Position
    ax2 = axes[0, 1]
    inv_locations = [i.location for i in fundamentals.inventories]
    inv_vs_avg = [i.vs_5yr_pct or 0 for i in fundamentals.inventories]
    colors2 = ['red' if x < 0 else 'green' for x in inv_vs_avg]
    ax2.barh(inv_locations, inv_vs_avg, color=colors2, alpha=0.7)
    ax2.axvline(x=0, color='black', linestyle='--', linewidth=1)
    ax2.set_xlabel('% vs 5-Year Average', fontsize=12)
    ax2.set_title('Inventory Position vs Historical', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='x')
    
    # Plot 3: Weekly Inventory Changes
    ax3 = axes[1, 0]
    inv_changes = [i.change_wow for i in fundamentals.inventories]
    colors3 = ['green' if x < 0 else 'red' for x in inv_changes]
    ax3.bar(inv_locations, inv_changes, color=colors3, alpha=0.7)
    ax3.axhline(y=0, color='black', linestyle='--', linewidth=1)
    ax3.set_ylabel('Million Barrels', fontsize=12)
    ax3.set_title('Weekly Inventory Changes', fontsize=14, fontweight='bold')
    ax3.tick_params(axis='x', rotation=45)
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Balance Summary
    ax4 = axes[1, 1]
    ax4.axis('off')
    
    summary_text = f"""
FUNDAMENTAL SUMMARY
{'='*40}

Date: {fundamentals.report_date}
Period: {fundamentals.data_period}

Supply: {balance.total_supply:.2f} million b/d
Demand: {balance.total_demand:.2f} million b/d
Balance: {balance.balance:+.2f} million b/d

Inventory Change: {balance.inventory_change:+.1f} MMbbl
Days of Supply: {balance.days_of_supply:.1f} days

{balance.interpretation}

KEY FACTORS:
{chr(10).join('• ' + f for f in fundamentals.qualitative_factors[:3])}
    """
    
    ax4.text(0.1, 0.9, summary_text, transform=ax4.transAxes,
             fontsize=10, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    plt.tight_layout()
    plt.show()


# Create visualization
visualize_fundamentals(fundamentals, balance)

## 6. Time Series Analysis of Fundamentals

In practice, you'd track fundamentals over time to identify trends.

In [ ]:
# Purpose: Create synthetic time series for fundamental analysis
# Key Concept: Tracking fundamental changes over time

def generate_synthetic_fundamental_series(num_weeks: int = 12) -> pd.DataFrame:
    """
    Generate synthetic fundamental data for demonstration.
    In production, this would come from historical EIA reports.
    
    Parameters
    ----------
    num_weeks : int
        Number of weeks to generate
        
    Returns
    -------
    pd.DataFrame
        Time series of fundamental metrics
    """
    # Generate dates
    end_date = datetime.now()
    dates = [end_date - timedelta(weeks=i) for i in range(num_weeks)]
    dates.reverse()
    
    # Generate synthetic data with trends
    base_production = 13.0
    base_demand = 20.0
    base_inventory = 440.0
    
    data = []
    for i, date in enumerate(dates):
        # Add seasonal patterns and noise
        seasonal = 0.3 * np.sin(2 * np.pi * i / 52)  # Annual seasonality
        noise = np.random.normal(0, 0.1)
        
        production = base_production + seasonal + noise
        demand = base_demand + 0.5 * seasonal + noise
        balance = production - demand
        inventory_change = -balance * 7  # Weekly change
        inventory = base_inventory + np.cumsum([inventory_change])[0]
        
        data.append({
            'date': date,
            'production': production,
            'demand': demand,
            'balance': balance,
            'inventory': inventory,
            'inventory_change': inventory_change,
            'days_supply': inventory / demand
        })
        
        base_inventory = inventory
    
    return pd.DataFrame(data)


# Generate synthetic time series
ts_data = generate_synthetic_fundamental_series()

# Display
print("Fundamental Time Series:")
print(ts_data.tail())

In [ ]:
# Purpose: Visualize fundamental trends over time
# Key Concept: Identifying directional changes in market balance

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Plot 1: Supply & Demand
ax1 = axes[0]
ax1.plot(ts_data['date'], ts_data['production'], label='Production', 
         marker='o', linewidth=2, color='green')
ax1.plot(ts_data['date'], ts_data['demand'], label='Demand', 
         marker='s', linewidth=2, color='red')
ax1.set_ylabel('Million b/d', fontsize=12)
ax1.set_title('Supply & Demand Trends', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Balance
ax2 = axes[1]
colors = ['red' if x < 0 else 'green' for x in ts_data['balance']]
ax2.bar(ts_data['date'], ts_data['balance'], color=colors, alpha=0.7)
ax2.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax2.set_ylabel('Million b/d', fontsize=12)
ax2.set_title('Supply/Demand Balance', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Plot 3: Inventory
ax3 = axes[2]
ax3.plot(ts_data['date'], ts_data['inventory'], 
         marker='o', linewidth=2, color='blue')
ax3.set_ylabel('Million Barrels', fontsize=12)
ax3.set_xlabel('Date', fontsize=12)
ax3.set_title('Inventory Level', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAverage Balance: {ts_data['balance'].mean():.2f} million b/d")
print(f"Balance Volatility: {ts_data['balance'].std():.2f} million b/d")
print(f"Average Days of Supply: {ts_data['days_supply'].mean():.1f} days")

## 7. Generating Forward Assessments

Use LLMs to synthesize current fundamentals into forward-looking views.

In [ ]:
# Purpose: Generate forward fundamental assessment using LLM
# Key Concept: Converting structured data into actionable insights

class FundamentalAnalyst:
    """LLM-powered fundamental analyst."""
    
    def __init__(self, api_key: Optional[str] = None):
        self.llm = ChatAnthropic(
            model="claude-3-5-sonnet-20241022",
            api_key=api_key or os.environ.get("ANTHROPIC_API_KEY"),
            temperature=0.3
        )
        
    def generate_assessment(self, 
                          fundamentals: CrudeFundamentals,
                          balance: SupplyDemandBalance,
                          recent_data: Optional[pd.DataFrame] = None) -> Dict[str, str]:
        """
        Generate forward-looking fundamental assessment.
        
        Parameters
        ----------
        fundamentals : CrudeFundamentals
            Current fundamental data
        balance : SupplyDemandBalance
            Supply/demand balance
        recent_data : pd.DataFrame, optional
            Historical trend data
            
        Returns
        -------
        dict
            Assessment with outlook, risks, and directional bias
        """
        # Build context from data
        context = f"""
CURRENT FUNDAMENTALS (as of {fundamentals.report_date}):

Supply/Demand Balance: {balance.balance:+.2f} million b/d
Inventory Change: {balance.inventory_change:+.1f} million barrels (WoW)
Days of Supply: {balance.days_of_supply:.1f} days

Production Data:
{json.dumps([asdict(p) for p in fundamentals.production], indent=2)}

Inventory Data:
{json.dumps([asdict(i) for i in fundamentals.inventories], indent=2)}

Demand Data:
{json.dumps([asdict(d) for d in fundamentals.demand], indent=2)}

Qualitative Factors:
{chr(10).join('- ' + f for f in fundamentals.qualitative_factors)}

Report Outlook: {fundamentals.outlook}
"""
        
        if recent_data is not None:
            context += f"""
RECENT TRENDS:
Average Balance (last 4 weeks): {recent_data['balance'].tail(4).mean():.2f} million b/d
Inventory Trend: {recent_data['inventory_change'].tail(4).sum():.1f} million barrels
"""
        
        # Create assessment prompt
        prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert oil market analyst. Provide a forward-looking 
fundamental assessment based on current data and trends.

Structure your response as:
1. SUMMARY: One-sentence market characterization
2. BULLISH FACTORS: Key supportive elements (2-3 bullets)
3. BEARISH FACTORS: Key negative elements (2-3 bullets)
4. OUTLOOK: 2-4 week forward view
5. DIRECTIONAL BIAS: Bullish/Neutral/Bearish with confidence

Be specific and quantitative where possible."""),
            ("user", "{context}")
        ])
        
        # Generate assessment
        chain = prompt | self.llm
        response = chain.invoke({"context": context})
        
        # Parse response sections
        content = response.content
        sections = {}
        current_section = None
        
        for line in content.split('\n'):
            if line.strip().endswith(':'):
                current_section = line.strip()[:-1].lower().replace(' ', '_')
                sections[current_section] = []
            elif current_section and line.strip():
                sections[current_section].append(line.strip())
        
        # Format output
        return {
            'full_assessment': content,
            'summary': '\n'.join(sections.get('summary', [])),
            'bullish_factors': '\n'.join(sections.get('bullish_factors', [])),
            'bearish_factors': '\n'.join(sections.get('bearish_factors', [])),
            'outlook': '\n'.join(sections.get('outlook', [])),
            'directional_bias': '\n'.join(sections.get('directional_bias', []))
        }


# Generate assessment
analyst = FundamentalAnalyst()
assessment = analyst.generate_assessment(fundamentals, balance, ts_data)

print("=== FUNDAMENTAL ASSESSMENT ===")
print(assessment['full_assessment'])

## Exercise 4.1: Extract and Analyze Your Own Report

Practice extracting fundamentals from a different commodity report.

In [ ]:
# Exercise: Create your own commodity report excerpt and extract fundamentals
# 
# Task:
# 1. Write a short report excerpt for a different commodity (natural gas, wheat, copper)
# 2. Extract fundamentals using the FundamentalExtractor
# 3. Calculate the supply/demand balance
# 4. Generate a forward assessment

# YOUR CODE HERE
# ---------------

# Step 1: Write your report excerpt
my_report = """
# Your commodity report text here...
"""

# Step 2: Extract fundamentals
my_fundamentals = None  # Replace with extraction

# Step 3: Calculate balance
my_balance = None  # Replace with calculation

# Step 4: Generate assessment
my_assessment = None  # Replace with assessment

In [ ]:
# SOLUTION - Run after attempting the exercise

solution_report = """
USDA Weekly Export Sales Report - December 21, 2023

Corn: Net sales of 1.2 million metric tons for 2023/24 were up 45% from the previous 
week and 23% from the prior 4-week average. Mexico purchased 425,000 MT, Japan 290,000 MT, 
and Colombia 185,000 MT. 

Current total commitments stand at 28.5 million MT, down 8% from last year at this time.

Ending stocks are projected at 2.1 billion bushels, representing 55 days of domestic use,
which is 5% below the 5-year average.

Key factors: Strong export demand from Mexico driven by livestock feed needs. Argentine 
production concerns supporting US export competitiveness. Brazil harvest progressing 
ahead of schedule, potentially pressuring prices in Q1.
"""

# Note: This would require adapting the CrudeFundamentals model for corn
# In practice, you'd create a CornFundamentals model or make the structure more generic

print("Solution approach:")
print("1. Create commodity-specific data models (CornFundamentals)")
print("2. Adapt extraction prompts for agricultural data")
print("3. Account for different units (MT vs barrels, bushels vs b/d)")
print("4. Consider seasonal patterns specific to crop cycles")

## Summary

### Key Takeaways

1. **Structured Extraction**: LLMs convert unstructured fundamental reports into structured data models
2. **Balance Calculation**: Supply/demand balances quantify market tightness
3. **Time Series Analysis**: Tracking fundamentals over time reveals trends
4. **Forward Assessment**: LLMs synthesize current data into actionable outlooks
5. **Multi-Modal Analysis**: Combine quantitative balance with qualitative factors

### What's Next

In **Module 4.2: Natural Gas Balance Model**, we'll:
- Build storage-driven models specific to natural gas
- Incorporate weather data and seasonal factors
- Model the supply/demand balance for a different commodity type

### Additional Resources

- [EIA Weekly Petroleum Status Report](https://www.eia.gov/petroleum/supply/weekly/)
- [IEA Oil Market Report](https://www.iea.org/reports/oil-market-report)
- "Oil 101" by Morgan Downey
- "The Prize" by Daniel Yergin

---

**Practice Exercise**: Apply this framework to the latest EIA WPSR and compare your extracted fundamentals with market price movements that week.